# Fase 1 — Selección de algoritmo: máscara neuronal vs geométrico

Primera fase de la narrativa. Se compara el beamformer **geométrico** (DS robusto
y MVDR-geo adaptativo) contra el **ciego** por máscara neuronal (**NM-MVDR**), y se
justifica elegir NM-MVDR por robustez. **El WPE ya no forma parte del sistema**
(`use_wpe=False` en todas las grillas).

- **P1 — alta diversidad acústica, sin errores.** Barrido amplio de RT60 × DOA-target
  × distancia × locutor × layout-interferentes × iSIR, con `error_*`/mismatch en 0.
  El geométrico recibe la posición **verdadera**, así que compite en igualdad → muestra
  que NM-MVDR al menos empata (y suele ganar con reverberación) en todo el envelope.
- **P2 — baja diversidad + errores de sensor/DOA.** Acústica fija; se barren por
  separado error de DOA, desajuste de ganancia y de fase. El geométrico se rompe
  (autocancelación por WNG alto / apuntado erróneo); NM-MVDR queda plano.

*Salida:* se elige **NM-MVDR** por robustez.

**Correr en orden:** Setup → Config → P1 → P2 → Figuras.

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [13]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

/content
fatal: destination path 'Vision-Aided-Beamformer' already exists and is not an empty directory.


In [9]:
import os

WHL = "/content/drive/MyDrive/colab_wheels"   # cache persistente de wheels en Drive

# CONSTRUIR (git+ para las libs de GitHub).
BUILD = [
    "noisereduce", "mir_eval", "pystoi", "pesq", "paderbox", "ai_edge_litert",
    "git+https://github.com/fgnt/pb_bss.git",
    "git+https://github.com/LCAV/pyroomacoustics.git",
    "git+https://github.com/fgnt/nara_wpe.git",
    "git+https://github.com/fakufaku/fast_bss_eval.git",
]
# INSTALAR desde cache: NOMBRES (no git+, si no pip vuelve a clonar).
INSTALL = [
    "noisereduce", "mir_eval", "pystoi", "pesq", "paderbox", "ai_edge_litert",
    "pb_bss", "pyroomacoustics", "nara_wpe", "fast_bss_eval",
]

# Reconstruye el cache SOLO si la lista de paquetes cambio (manifest) -> se
# autocura si agrego/saco un paquete, sin tener que borrar el cache a mano.
manifest = os.path.join(WHL, ".manifest.txt")
key = "\n".join(sorted(BUILD))
need_build = (not os.path.isfile(manifest)) or open(manifest).read() != key

if need_build:
    os.makedirs(WHL, exist_ok=True)
    print("[*] (Re)construyendo cache de wheels en Drive (una vez por cambio de lista)...")
    !pip wheel --wheel-dir=$WHL {" ".join(BUILD)}
    with open(manifest, "w") as fh:
        fh.write(key)
    print("[*] Cache actualizado en", WHL)

!pip install --no-index --find-links=$WHL {" ".join(INSTALL)}
print("[*] Paquetes instalados desde el cache de Drive.")
# Si Colab actualiza Python y falla un import:  !rm -rf $WHL  (se reconstruye solo)

[*] (Re)construyendo cache de wheels en Drive (una vez por cambio de lista)...
  Cloning https://github.com/fgnt/pb_bss.git to /tmp/pip-req-build-sin7vh57
  Running command git clone --filter=blob:none --quiet https://github.com/fgnt/pb_bss.git /tmp/pip-req-build-sin7vh57
  Resolved https://github.com/fgnt/pb_bss.git to commit 10acc347fc9ea21e3d312806a0bd751d0d0af183
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/LCAV/pyroomacoustics.git to /tmp/pip-req-build-47opb6dr
  Running command git clone --filter=blob:none --quiet https://github.com/LCAV/pyroomacoustics.git /tmp/pip-req-build-47opb6dr
  Resolved https://github.com/LCAV/pyroomacoustics.git to commit ff7d61f219e4eb41489963c4bb5f57bea5bc2c69
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproj

[*] Paquetes instalados desde el cache de Drive.


In [10]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

/content/Vision-Aided-Beamformer
From https://github.com/MatiasVereert/Vision-Aided-Beamformer
 * branch            main       -> FETCH_HEAD
Already up to date.


## Config compartida (Fase 1)

In [ ]:
import sys, os, numpy as np, shutil
from datetime import datetime

repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')
for p in (repo_root, src_path):
    if p not in sys.path: sys.path.append(p)
%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    TFLITE_AVAILABLE = False

from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import DS, MVDR_Recursive, NM_MVDR, MPDR_Recursive
from propagation.mird_loader import MirdDatasetProvider

m1 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
m2 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")
interpreter_1 = interpreter_2 = None
if TFLITE_AVAILABLE and os.path.exists(m1) and os.path.exists(m2):
    interpreter_1 = tf.lite.Interpreter(model_path=m1); interpreter_1.allocate_tensors()
    interpreter_2 = tf.lite.Interpreter(model_path=m2); interpreter_2.allocate_tensors()
    print("[*] DTLN TFLite OK.")
else:
    print("[!] Sin DTLN interpreters.")

input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"
mird_dir  = "/content/drive/MyDrive/Benchmarks_tesis/rirs"
provider = MirdDatasetProvider(root_dir=mird_dir)

# ===================== PERILLAS =====================
DURATION = 15
# ===================================================

TARGETS = [os.path.join(input_dir, f) for f in [
    "p002_emo_adoration_sentences.wav",
    "p008_emo_contentment_sentences.wav",
]]
INTERF = [os.path.join(input_dir, f) for f in [
    "techno_gated commune.wav",
    "hairdryer_07_SH_MKH800.wav",
    "drill_07_RHODE_NT1.wav",
]]

base_config = {
    'fs': 16000, 'duration': DURATION, 't_early': 0.050,
    'array_center': [3.0, 3.0, 1.2], 'mird_spacing': "3-3-3-8-3-3-3",
    'snr_db': 60.0,
    'source_path': TARGETS[0], 'interf_paths': INTERF,
    # --- WPE ELIMINADO DEL SISTEMA (use_wpe=False en toda grilla). Estos escalares
    #     solo existen porque el benchmark los exige en scene_base_config; no operan.
    'wpe_taps': 5, 'wpe_delay': 2, 'wpe_alpha': 0.9999,
    'wpe_stft_size': 512, 'wpe_stft_shift': 128,
    'stft_window': 512, 'stft_overlap': 384,
    'dtln_model_path': m1,
    'eval_references': ['early'],
}

# --- PROCESADORES (Fase 1: la comparación de algoritmo) ---
# DS = geométrico robusto (WNG bajo); MVDR-geo = geométrico adaptativo (frágil);
# NM-MVDR = máscara neuronal (ciego). Hiperparámetros con defaults; ajustables.
processors_dict = {
    "DS":       DS(),
    "MVDR-geo": MVDR_Recursive(min_loading=1e-6),
    "NM-MVDR":  NM_MVDR(min_loading=1e-6, alpha=0.99),
    "MPDR": MPDR_Recursive(min_loading=1e-6),
}

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M")
def run(grid, name, procs=processors_dict):
    t = f"/content/results_temp/F1_{name}_{RUN_TAG}"
    d = f"/content/drive/MyDrive/Tesis_Beamformers/results/F1_{name}_{RUN_TAG}"
    os.makedirs(t, exist_ok=True); os.makedirs(d, exist_ok=True)
    df = run_mird_grid_search(grid_params=grid, dataset_provider=provider,
                              processors=procs, scene_base_config=base_config,
                              output_dir=t, interpreter_1=interpreter_1,
                              interpreter_2=interpreter_2, save_catalog=False)
    shutil.copytree(t, d, dirs_exist_ok=True)
    print(f"[EXITO] {name} -> {d}")
    return df, d

print("Config Fase 1 lista. procesadores =", list(processors_dict.keys()))

/content/Vision-Aided-Beamformer/src
[*] DTLN TFLite OK.
[MirdDatasetProvider] Successfully indexed 234 RIR files from: /content/drive/MyDrive/Benchmarks_tesis/rirs
Config Fase 1 lista. procesadores = ['DS', 'MVDR-geo', 'NM-MVDR', 'MPDR']


## P1 — alta diversidad acústica (resultado general, sin errores)

In [15]:
# Barre RT60 x DOA-target x distancia x locutor x layout-interf x iSIR, con
# mismatch y error_* en 0. Grilla marginalizada: recortá listas si tarda mucho.
# MIRD disponible: RT60 {0.160,0.360,0.610}, dist {1,2} m, ángulos ±90 en pasos de 15°.
INTERF_P1 = [
    [(45, 1.0, 0)],                                   # 1 interferente
    [(30, 1.0, 0), (-45, 1.0, 1)],                    # 2 interferentes
    [(30, 1.0, 0), (-30, 1.0, 1), (60, 1.0, 2)],      # 3 interferentes
]
grid_P1 = dict(
    rt60=[0.160, 0.360, 0.610],
    target_angle=[0, 30, 60],
    target_dist=[1.0, 2.0],
    source_path=TARGETS,
    interf_configs=INTERF_P1,
    isir_db=[-5, 0, 5, 10],
    use_wpe=[False],
    mismatch_gain=[0], mismatch_phase=[0],
    error_angle_deg=[0.0], error_distance_m=[0.0],
)
df_P1, dir_P1 = run(grid_P1, "P1_diversidad")

[*] t_early FIXED at 8.0 ms (decoupled from wpe_delay).
[*] Total experiments to run: 432 per processor.


Running MIRD Benchmark:   0%|          | 0/432 [00:00<?, ?exp/s]


--- Iteration 1/432 | Config: {'rt60': 0.16, 'target_angle': 0, 'target_dist': 1.0, 'source_path': '/content/drive/MyDrive/Benchmarks_tesis/inputs/p002_emo_adoration_sentences.wav', 'interf_configs': [(30, 1.0, 0), (-30, 1.0, 1), (60, 1.0, 2)], 'isir_db': -5, 'use_wpe': False, 'mismatch_gain': 0, 'mismatch_phase': 0, 'error_angle_deg': 0.0, 'error_distance_m': 0.0, 'wpe_taps': 5, 'wpe_delay': 2, 'wpe_method': 'online', 'wpe_block_L': 512, 'wpe_block_shift': 64} ---
 -> [NODE 1] Physical setup changed. Extracting MIRD RIRs...
[Loader] Remuestreando de 48000 Hz a 16000 Hz...
[Loader] Audio corto, repitiendo en bucle para llenar duración...


/content/Vision-Aided-Beamformer/src/utils/audio.py:32: WavFileWarning: Chunk (non-data) not understood, skipping it.
  fs_file, data = wavfile.read(filename)


[Loader] Remuestreando de 44100 Hz a 16000 Hz...
[Loader] Remuestreando de 48000 Hz a 16000 Hz...
[Loader] Audio corto, repitiendo en bucle para llenar duración...
[Loader] Remuestreando de 48000 Hz a 16000 Hz...
[Loader] Audio corto, repitiendo en bucle para llenar duración...
[SimAcoustic: MIRD] Mapping Target Source -> Snapped to Grid: Dist=1.0m, Angle=0°
[SimAcoustic: MIRD] Mapping Interf #1 -> Snapped to Grid: Dist=1.0m, Angle=30°
[SimAcoustic: MIRD] Mapping Interf #2 -> Snapped to Grid: Dist=1.0m, Angle=-30°
[SimAcoustic: MIRD] Mapping Interf #3 -> Snapped to Grid: Dist=1.0m, Angle=60°
[SimAcoustic] Triggering high-fidelity RIR resampling: 48000 Hz -> 16000 Hz
[SimAcoustic] Successfully loaded and synchronized real dataset environment spanning 8 sensor channels.


...

   -> Processing with: MVDR-geo (ErrAng: 0.0deg, ErrDist: 0.0m)...


Running MIRD Benchmark:   0%|          | 2/432 [00:55<2:42:27, 22.67s/exp]

   -> [NODE 6] Applying DTLN post MVDR-geo...


Running MIRD Benchmark:   0%|          | 2/432 [00:57<3:24:47, 28.58s/exp]


KeyboardInterrupt: 

### Preview rápido P1 (en memoria — no necesita las corridas de P2)

In [ ]:
# Boxplots Δ por procesador sobre la diversidad de P1. Usa df_P1 en memoria.
import matplotlib.pyplot as plt, seaborn as sns
_pr = ["DS", "MVDR-geo", "NM-MVDR"]
_c  = {"DS":"tab:green", "MVDR-geo":"tab:red", "NM-MVDR":"tab:orange"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (col, lbl) in zip(axes, [("Delta_tot_PESQ_early","Δ PESQ"),
                                 ("Delta_tot_SIR_early","Δ SIR [dB]")]):
    if col not in df_P1.columns: continue
    sns.boxplot(data=df_P1, x="processor", y=col, order=_pr, palette=_c, ax=ax, fliersize=1)
    ax.set_xlabel(""); ax.set_ylabel(lbl); ax.grid(alpha=0.3, axis="y")
plt.suptitle("Preview P1 — Δ por procesador (alta diversidad acústica)")
plt.tight_layout(); plt.show()

## P2 — baja diversidad + errores de sensor/DOA (robustez)

In [ ]:
# Acústica FIJA (RT60=360 ms, target broadside 0/1 m, iSIR=0, un layout de 3 interf).
# Tres barridos 1D independientes: error de DOA | ganancia | fase.
FIXED_P2 = dict(
    rt60=[0.360], target_angle=[0], target_dist=[1.0],
    source_path=TARGETS,
    interf_configs=[[(30, 1.0, 0), (-15, 1.0, 1), (45, 1.0, 2)]],
    isir_db=[0], use_wpe=[False],
)
# (a) error de DOA -> solo afecta a DS y MVDR-geo (usan source_pos); NM-MVDR plano.
grid_P2_doa = dict(FIXED_P2,
    mismatch_gain=[0], mismatch_phase=[0],
    error_angle_deg=[0, 2, 5, 10, 15], error_distance_m=[0.0])
# (b) desajuste de GANANCIA entre sensores.
grid_P2_gain = dict(FIXED_P2,
    error_angle_deg=[0.0], error_distance_m=[0.0],
    mismatch_gain=[0, 1, 2, 3], mismatch_phase=[0])
# (c) desajuste de FASE entre sensores.
grid_P2_phase = dict(FIXED_P2,
    error_angle_deg=[0.0], error_distance_m=[0.0],
    mismatch_gain=[0], mismatch_phase=[0, 3, 6, 10])

df_P2_doa,   dir_P2_doa   = run(grid_P2_doa,   "P2_doa")
df_P2_gain,  dir_P2_gain  = run(grid_P2_gain,  "P2_gain")
df_P2_phase, dir_P2_phase = run(grid_P2_phase, "P2_phase")

### Preview rápido P2 (curvas de robustez, en memoria)

In [ ]:
# Δ SIR vs cada barrido (DOA / ganancia / fase). Usa los df_P2_* en memoria.
import matplotlib.pyplot as plt
_pr = ["DS", "MVDR-geo", "NM-MVDR"]
_c  = {"DS":"tab:green", "MVDR-geo":"tab:red", "NM-MVDR":"tab:orange"}
def _prev(df, xcol, xlabel, ax):
    for pr in _pr:
        s = df[df.processor == pr]
        if s.empty: continue
        g = s.groupby(xcol)["Delta_tot_SIR_early"].mean()
        ax.plot(g.index.values, g.values, "-o", color=_c[pr], label=pr)
    ax.set_xlabel(xlabel); ax.set_ylabel("Δ SIR [dB]"); ax.grid(alpha=0.3)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
_prev(df_P2_doa,   "error_angle_deg", "error DOA [°]",   axes[0])
_prev(df_P2_gain,  "mismatch_gain",   "ganancia [dB]",   axes[1])
_prev(df_P2_phase, "mismatch_phase",  "fase [°]",        axes[2])
axes[0].legend(fontsize=8)
plt.suptitle("Preview P2 — Δ SIR vs error de DOA / mismatch de sensor")
plt.tight_layout(); plt.show()

## Figuras — P1 (envelope general) y P2 (curvas de robustez)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns

METR  = [("Delta_tot_PESQ_early","Δ PESQ"), ("Delta_tot_STOI_early","Δ STOI"),
         ("Delta_tot_SDR_early","Δ SDR [dB]"), ("Delta_tot_SIR_early","Δ SIR [dB]")]
PROCS = ["DS", "MVDR-geo", "NM-MVDR"]
PAL   = {"DS":"tab:green", "MVDR-geo":"tab:red", "NM-MVDR":"tab:orange"}

# --- P1: boxplots marginales por procesador (dispersión = incertidumbre real) ---
dfp1 = pd.read_csv(os.path.join(dir_P1, "mird_benchmark_metrics.csv"))
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax,(col,lbl) in zip(axes.ravel(), METR):
    if col not in dfp1.columns: continue
    sns.boxplot(data=dfp1, x="processor", y=col, order=PROCS, palette=PAL, ax=ax, fliersize=1)
    ax.set_xlabel(""); ax.set_ylabel(lbl); ax.grid(alpha=0.3, axis="y")
fig.suptitle("Fase 1 · P1 — Δ sobre alta diversidad acústica")
fig.tight_layout(); fig.savefig(os.path.join(dir_P1,"F1_P1_box.png"), dpi=140, bbox_inches="tight"); plt.show()

# tendencia media vs RT60 (SDR/SIR)
fig2, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax,(col,lbl) in zip(axes, METR[2:]):
    for pr in PROCS:
        g = dfp1[dfp1.processor==pr].groupby("rt60")[col].mean()
        ax.plot(g.index.values*1000, g.values, "-o", color=PAL[pr], label=pr)
    ax.set_xlabel("RT60 [ms]"); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
axes[0].legend()
fig2.suptitle("Fase 1 · P1 — tendencia vs RT60"); fig2.tight_layout()
fig2.savefig(os.path.join(dir_P1,"F1_P1_rt.png"), dpi=140, bbox_inches="tight"); plt.show()

# --- P2: curvas 1D (el cruce geométrico -> ciego) ---
def plot_curves(dir_csv, xcol, xlabel, title, fname):
    df = pd.read_csv(os.path.join(dir_csv, "mird_benchmark_metrics.csv"))
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax,(col,lbl) in zip(axes.ravel(), METR):
        if col not in df.columns: continue
        for pr in PROCS:
            sub = df[df.processor==pr]
            if sub.empty: continue
            g = sub.groupby(xcol)[col]; m, s = g.mean(), g.std()
            ax.plot(m.index.values, m.values, "-o", color=PAL[pr], ms=5, label=pr)
            ax.fill_between(m.index.values, (m-s).values, (m+s).values, color=PAL[pr], alpha=0.12)
        ax.set_xlabel(xlabel); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
    axes.ravel()[0].legend(fontsize=8)
    fig.suptitle(title); fig.tight_layout()
    fig.savefig(os.path.join(dir_csv, fname), dpi=140, bbox_inches="tight"); plt.show()

if all(v in globals() for v in ["dir_P2_doa", "dir_P2_gain", "dir_P2_phase"]):
    plot_curves(dir_P2_doa,   "error_angle_deg", "error DOA [°]",           "Fase 1 · P2 — Δ vs error de DOA",  "F1_P2_doa.png")
    plot_curves(dir_P2_gain,  "mismatch_gain",   "desajuste ganancia [dB]", "Fase 1 · P2 — Δ vs ganancia",     "F1_P2_gain.png")
    plot_curves(dir_P2_phase, "mismatch_phase",  "desajuste fase [°]",      "Fase 1 · P2 — Δ vs fase",         "F1_P2_phase.png")
else:
    print("[i] Corré P2 (doa/gain/phase) para las curvas de robustez de esta figura.")